In [ ]:
!pip install ultralytics

In [ ]:
import os
import random
import shutil
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import cv2

In [ ]:
data_root = '/kaggle/input/cigarette-smoker-detection/data/'
print(os.listdir(data_root))

In [ ]:
image = cv2.imread('/kaggle/input/cigarette-smoker-detection/data/not_smoking/000015.jpg')
if image is None or image.size < 0:
    print(f"Ошибка: Не удалось прочитать изображение")

In [ ]:
!ls /kaggle/input/cigarette-smoker-detection/data

In [ ]:
dest_dir = '/kaggle/working/'
os.makedirs(os.path.join(dest_dir, 'data'))

In [ ]:
train_path = '/kaggle/working/data/train'
val_path = '/kaggle/working/data/val'

In [ ]:
class1_train_path = os.path.join(train_path, 'smoking')
class2_train_path = os.path.join(train_path, 'not_smoking')
class1_val_path = os.path.join(val_path, 'smoking')
class2_val_path = os.path.join(val_path, 'not_smoking')

In [ ]:
os.makedirs(class1_train_path, exist_ok=True)
os.makedirs(class2_train_path, exist_ok=True)
os.makedirs(class1_val_path, exist_ok=True)
os.makedirs(class2_val_path, exist_ok=True)


In [ ]:
smoking_files = []
not_smoking_files = []

for foldername in os.listdir(data_root):
    folder_path = os.path.join(data_root, foldername)
    if os.path.isdir(folder_path):
        if foldername == 'smoking':
            smoking_files += [os.path.join(folder_path, filename) for filename in os.listdir(folder_path)]
        elif foldername == 'not_smoking':
            not_smoking_files += [os.path.join(folder_path, filename) for filename in os.listdir(folder_path)]

In [ ]:
class1_train_files, class1_val_files = train_test_split(smoking_files, test_size=0.2, random_state=42, shuffle=True)
class2_train_files, class2_val_files = train_test_split(not_smoking_files, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
# Копирование файлов класса 1 в папку train
for filepath in class1_train_files:
    filename = os.path.basename(filepath)
    shutil.copy2(filepath, os.path.join(class1_train_path, filename))

# Копирование файлов класса 1 в папку val
for filepath in class1_val_files:
    filename = os.path.basename(filepath)
    shutil.copy2(filepath, os.path.join(class1_val_path, filename))

# Копирование файлов класса 2 в папку train
for filepath in class2_train_files:
    filename = os.path.basename(filepath)
    shutil.copy2(filepath, os.path.join(class2_train_path, filename))

# Копирование файлов класса 2 в папку val
for filepath in class2_val_files:
    filename = os.path.basename(filepath)
    shutil.copy2(filepath, os.path.join(class2_val_path, filename))

In [ ]:
def check_images_in_directory(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            image = cv2.imread(file_path)
            if image is None or image.size <= 0:
                #print(f"Ошибка: Не удалось прочитать изображение {file_path}")
                os.remove(file_path)

directory_path = '/kaggle/working/data'
check_images_in_directory(directory_path)

In [ ]:
model = YOLO("yolov8x-cls.pt")

In [ ]:
model.train(data="/kaggle/working/data", epochs=20, imgsz=640, batch=16, optimizer='SGD', seed=42,
           dropout=0.3)

In [ ]:
tt = YOLO("/kaggle/working/runs/classify/train/weights/best.pt")

In [ ]:
tt.val()

In [ ]:
tt.export(format="onnx")

In [ ]:
!rm -rf data